In [1]:
!pip install pgmpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pgmpy]


In [1]:
import warnings
warnings.filterwarnings("ignore")   # hide library notices

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

# 1) The graph: each pair is an arrow (parent, child)
model = DiscreteBayesianNetwork([
    ("rain", "maintenance"),
    ("rain", "train"),
    ("maintenance", "train"),
    ("train", "appointment"),
])

# 2) P(Rain): no parents, so one column
rain = TabularCPD(
    variable="rain", variable_card=3,
    values=[[0.7],    # none
            [0.2],    # light
            [0.1]],   # heavy
    state_names={"rain": ["none", "light", "heavy"]})

# 3) P(Maintenance | Rain): one column for each value of rain
#                rain = none  light  heavy
maintenance = TabularCPD(
    variable="maintenance", variable_card=2,
    values=[[0.4, 0.2, 0.1],    # yes
            [0.6, 0.8, 0.9]],   # no
    evidence=["rain"], evidence_card=[3],
    state_names={"maintenance": ["yes", "no"],
                 "rain": ["none", "light", "heavy"]})

# 4) P(Train | Rain, Maintenance): one column for each (rain, maintenance) pair
#     (none,yes) (none,no) (light,yes) (light,no) (heavy,yes) (heavy,no)
train = TabularCPD(
    variable="train", variable_card=2,
    values=[[0.8, 0.9, 0.6, 0.7, 0.4, 0.5],    # on time
            [0.2, 0.1, 0.4, 0.3, 0.6, 0.5]],   # delayed
    evidence=["rain", "maintenance"], evidence_card=[3, 2],
    state_names={"train": ["on time", "delayed"],
                 "rain": ["none", "light", "heavy"],
                 "maintenance": ["yes", "no"]})

# 5) P(Appointment | Train): one column for each value of train
#                train = on time  delayed
appointment = TabularCPD(
    variable="appointment", variable_card=2,
    values=[[0.9, 0.6],    # attend
            [0.1, 0.4]],   # miss
    evidence=["train"], evidence_card=[2],
    state_names={"appointment": ["attend", "miss"],
                 "train": ["on time", "delayed"]})

# 6) Put the tables into the graph and check them
model.add_cpds(rain, maintenance, train, appointment)
print("Model is correct:", model.check_model())

Model is correct: True


In [2]:
# P(r, m, t, a) = P(r) x P(m | r) x P(t | r, m) x P(a | t)
p = (rain.get_value(rain="none")
     * maintenance.get_value(maintenance="no", rain="none")
     * train.get_value(train="on time", rain="none", maintenance="no")
     * appointment.get_value(appointment="attend", train="on time"))
print(round(p, 4))

0.3402


In [3]:
from pgmpy.inference import VariableElimination
inference = VariableElimination(model)

def show(evidence):
    """Print P(X | evidence) for every node X that we did not observe."""
    for node in ["rain", "maintenance", "train", "appointment"]:
        if node in evidence:
            print(f"{node}: {evidence[node]}   (observed)")
            continue
        result = inference.query([node], evidence=evidence, show_progress=False)
        print(node)
        for value, prob in zip(result.state_names[node], result.values):
            print(f"    {value}: {prob:.4f}")

show({"train": "delayed"})

rain
    none: 0.4601
    light: 0.3005
    heavy: 0.2394
maintenance
    yes: 0.3662
    no: 0.6338
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000


In [4]:
show({"appointment": "miss"})

rain
    none: 0.6065
    light: 0.2392
    heavy: 0.1544
maintenance
    yes: 0.3441
    no: 0.6559
train
    on time: 0.4802
    delayed: 0.5198
appointment: miss   (observed)


In [6]:
# ### Lab Short Answers

# * Question 13: The hand calculation and the code results match perfectly because the pgmpy library uses exact mathematical inference algorithms (like Variable Elimination) to calculate the values rather than using random sampling or approximations.
# * Question 14: The value is exactly 0.4000 because the Appointment node depends directly and only on the Train node. Since the train status is given as an observed fact, the upstream variables (rain and maintenance) are blocked, allowing us to read the probability directly from the CPT table.
# * Question 16: The probability of heavy rain went up because of diagnostic reasoning flowing backwards. Knowing that an appointment was missed implies the train was likely delayed, and a train delay increases the likelihood that it rained heavily.
# * Question 18: The change is very minor because heavy rain on its own already accounts for a 50% chance of a train delay, even without track maintenance. Since a delay is highly probable under heavy rain anyway, observing a delay provides very little additional proof that maintenance occurred.
